In [ ]:
!pip install fredapi
!pip install yfinance
!pip install yahoo_fin

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 9.9 MB/s eta 0:00:00


In [ ]:
from fredapi import Fred as fd
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from datetime import date
from datetime import datetime as dt
from yahoo_fin import options
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
fred = fd(api_key='Put your own key')

# Scenario 1: Credit cards

In [ ]:
# Let's get credit card interest rate and FED interest rate over the same period
credit_card_rate = fred.get_series('TERMCBCCALLNS', observation_start='1994-01-01', observation_end='2024-05-01')
fed_rate = fred.get_series('FEDFUNDS', observation_start='1994-01-01', observation_end='2024-05-01')

In [ ]:
plt.hist(credit_card_rate)
plt.xlabel('Rate (%)')
plt.title('Commercial banks Credit card interest rate distribution')

> Credit card interest rate is mostly between 12 - 18% annually. In some rare cases, it can reach as high as 22% per year

In [ ]:
# Fill NaN value in credit_card_rate by the previous non-null value of the first month in the same quarter
credit_card_rate.ffill(inplace=True)

# Merge credit card and FED interest rate to compare
df = pd.DataFrame({'CC_Rate': credit_card_rate, 'FED_Rate': fed_rate})

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(14, 7)
ax.plot(credit_card_rate.index, credit_card_rate.values)
ax.plot(fed_rate.index, fed_rate.values)

ax.set_xlabel('Time')
ax.set_ylabel('Rate (%)')
ax.set_title('Commercial banks Credit card and FED interest rates')

In [ ]:
# Calculate correlation between credit card and FED interest rate
df['CC_Rate'].corr(df['FED_Rate'], method="spearman")


> Credit card interest rate and FED interest rate have quite high correlation coefficient and vary in a positive relationship. The reason is when FED's rate changes, credit card interest rate must be adjusted accordingly so that it is profitable for the banks and also attractive for borrower

# Scenario 2: Mortgages

In [ ]:
# Let's get 15-Year Fixed Rate Mortgage Average in the United States data
mortgages_rate = fred.get_series('MORTGAGE15US')

In [ ]:
plt.plot(mortgages_rate.index, mortgages_rate.values)
plt.xlabel('Time')
plt.ylabel('Mortgage Rate (%)')
plt.title('US Mortgages Rate Over Time')

In [ ]:
mortgages_delinquency_rate = fred.get_series('DRSFRMACBS')
plt.plot(mortgages_delinquency_rate.index, mortgages_delinquency_rate.values)
plt.xlabel('Time in years')
plt.ylabel('Delinquency Rate (%)')
plt.title('US Mortgages Delinquency rate VS time')

In [ ]:
# Calculate the correlation between mortgage rate and delinquency rate
resampled_mortgages_rate = mortgages_rate.resample('Q').mean()
resampled_mortgages_rate.reset_index(drop=True).corr(mortgages_delinquency_rate[1:].reset_index(drop=True), method="spearman")


> Mortgage rate and delinquency rate have no correlation. So it is safe to assume that mortgage rate has no effect on defaults or delinquency. The only noticeable thing about delinquency rate is a peak from 2008 to 2012, after the Great Financial Crisis

# Scenario 3: Money at a fixed rate for a business for a construction loan.


In [ ]:
# Let's get Business loan delinquency rate and FED interest rate over the same period
delinquency_loan_rate = fred.get_series('DRBLACBS', observation_start='1996-01-01', observation_end='2024-05-01')
fed_rate = fred.get_series('FEDFUNDS', observation_start='1996-01-01', observation_end='2024-05-01')

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(14, 7)
ax.plot(delinquency_loan_rate.index, delinquency_loan_rate.values, label="Delinquency Rate")
ax.plot(fed_rate.index, fed_rate.values, label="Interest Rate")
ax.set_xlabel('Time')
ax.set_ylabel('Rate (%)')
ax.set_title('Business Loan Delinquency Rate and FED interest rates, from 1996 to 2004')
ax.legend()


> From 2000 to 2012, we can observe a relationship between business loan delinquency and central bank interest rate. Delinquency rate peaks lag with interest rate peaks by a period of about 2 years. However, since 2020, we can see a surge in interest rate but the delinquency rate shows little changes. So we cannot conclude a clear causal or correllation between interest rate and business loan delinquency


In [ ]:
# Distribution of delinquency rate changes

import matplotlib.pyplot as plt

# Distribution of delinquency rate changes
delinquency_rate_diff = delinquency_loan_rate.diff().dropna()

# Create a figure and axis for the histogram
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the histogram
ax.hist(delinquency_rate_diff, bins=30, color='green', edgecolor='black')
ax.set_xlabel('Change in Delinquency Rate (%)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Changes in Business Loan Delinquency Rate')

plt.tight_layout()
plt.show()


> There is a slight skewness to the right indicating that the distribution of Changes in Business Loan Delinquency Rate is slightly positively skewed. Most data points are around the mean.


#Scenario 4: AAPL equity



In [ ]:
'''Use Python to graph series, distributions, 2-way relationships, and any other meaningful
visuals that show insight into the data. Your group will decide both what to graph, and
write the code to create them. Each visual must include 1 sentence that must explicitly
describe the insight.
'''
vix = yf.download("^VIX", period="5y", interval="1d") # downloading the VIX data

aapl = yf.download('aapl', period="5y", interval="1d") # Downloading the equity

spy = yf.download('spy', period="5y", interval="1d") #Downloading the SP500 as a representative of the market



In [ ]:
aapl['ret']=aapl['Close'].pct_change()*100
vix['ret']=vix['Close'].pct_change()*100
spy['ret']=spy['Close'].pct_change()*100
spy.dropna(inplace=True)
vix.dropna(inplace=True)
aapl.dropna(inplace=True)

In [ ]:
plt.hist(aapl['ret'],bins=100)
plt.xlabel('Returns of AAPL %')
plt.ylabel('Frequency')
plt.title('AAPL Returns % VS Frequency')
plt.show()

#We see there is a slgiht skewness to the right but all we can say about the return distribution is if it is Gaussian or not within a specific window of confidence so we start by using Kolmogrov-Smirnov Normality test.

In [ ]:
ks_statistic, p_value = stats.kstest(aapl['ret'], 'norm', args=(aapl['ret'].mean(), aapl['ret'].std()))

print(f"K-S Statistic: {ks_statistic}")
print(f"P-value: {p_value}")

#We see that the data are not normally distibuted which is expected as return follow a power-law and have more inclination to go up due to inflation

In [ ]:
spy['sret']=(spy['ret']-spy['ret'].mean())/spy['ret'].std()
aapl['sret']=(aapl['ret']-aapl['ret'].mean())/aapl['ret'].std() #standardizing the returns for both return series whether it is spy or aapl
vix['mean_vol']=vix['Close'].rolling(window=20).mean()

vix['smean_vol']=(vix['mean_vol']-vix['mean_vol'].mean())/vix['mean_vol'].std()

aapl['stdev']=aapl['ret'].rolling(20).std()# monthly volatility (20 trading days)
aapl.dropna(inplace=True)

aligned_data = pd.DataFrame({'aapl_sret': aapl['sret'], 'spy_sret': spy['sret']}).dropna() #putting them in the same data frame



aligned_data['corr'] = aligned_data['aapl_sret'].rolling(window=20).corr(aligned_data['spy_sret']).rolling(window=20).mean()


fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 6))

# Plot 1: Rolling 20-day mean correlation between AAPL and SPY
axes[0].plot(aligned_data.index, aligned_data['corr'], label='20-Day Rolling Mean Correlation', color='blue')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Mean 20-day Correlation')
axes[0].set_title('20-Day Rolling Mean Correlation Between AAPL and SPY Returns')
axes[0].grid(True)
axes[0].legend()

# Plot 2: VIX standardized mean value over time
axes[1].plot(vix.index, vix['smean_vol'], label='VIX standardized average value in 20-days', color='red')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('VIX standardized 20-day mean')
axes[1].set_title('VIX 20-day standardized mean value Over Time')
axes[1].grid(True)
axes[1].legend()


axes[2].plot(aapl.index, aapl['stdev'], label='20-day Standard Deviation of AAPL value ', color='green')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('20-day Standard Deviation of AAPL value')
axes[2].set_title('20-day Standard Deviation of AAPL value Over Time')
axes[2].grid(True)
axes[2].legend()
# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

#On the upper left, we see that the correlation between the market returns and apple stock returns are volatile, on the right we also notice that this correlation is also going up when the VIX goes up (Middle graph), at least within the averaging period of 20-days or once month of trading. Finally, we see that the moving standard deviation of apple goes up as well once the correlation goes up and when vix goes up. indication some sort of correlation between them (Right most plot).

In [ ]:
plt.hist(vix['ret'],bins=100)
plt.xlabel('Change of VIX %')
plt.ylabel('Frequency')
plt.title('VIX change % VS Frequency')
plt.show()

#It seems that the VIX does change more dramatically to the right side (more upward extremes than downward) we do not claim anything on the distribution.

In [ ]:
data = pd.DataFrame({'aapl_ret': aapl['ret'], 'vix_ret': vix['ret']}).dropna()
data.describe()

In [ ]:
X = data[['vix_ret']]
y = data['aapl_ret']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=99)

model=LinearRegression()
model.fit(X_train,y_train)
print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_[0]}")
y_pred = model.predict(X_test)

# Calculate mean squared error and R-squared
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)



print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")
plt.figure(figsize=(8, 6))
plt.scatter(X_test, y_test, color='blue', label='Actual AAPL Returns')
plt.plot(X_test, y_pred, color='red', label='Predicted AAPL Returns' , linewidth=2)
plt.xlabel('Change in Volatility of the Market (VIX %)')
plt.ylabel('AAPL Returns %')
plt.title('Linear Regression: VIX Returns vs AAPL Returns')
plt.legend()
plt.grid(True)
plt.show()

#Although linear regression requires no outliers as an assumption, but clearly there is a negative relationship between market volatility and negative returns of stocks (In this case AAPL). In this simple model we find that market volatility explains 19% of stock returns in this model. which significant considering that there are so many other factors that get into variability of returns from Macro news to micro news. We also use Speaman correltion to catch non-linear correlation and it shows a good negative correlation between the stock and volatility

In [ ]:
data['vix_ret'].corr(data['aapl_ret'],method='spearman') #Spearman correlation

In [ ]:
data['vix_ret'].corr(data['aapl_ret'],method='kendall') #kenadall correlation

In [ ]:
data['vix_ret'].corr(data['aapl_ret']) # Pearson correlation

#Due to high concentration of volatility changes around 0 we see a cluster, however, there is so sort of a relationship between high VIX changes and lower returns. We plot a linear regression line between them and show that VIX can explain

In [ ]:
#Finally we repeat the analysis but with a shifted period of one day to see if there is some sort of causal relationship
shift_period = 1  # We can change this to explore different shift periods
data['aapl_future_ret'] = data['aapl_ret'].shift(-shift_period)  # Future aapl returns
shifted_data = data.dropna()

# Define independent (X) and dependent (y) variables
X = shifted_data[['vix_ret']]  # VIX returns as independent variable
y = shifted_data['aapl_future_ret']  # Future AAPL returns as dependent variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=99)
model = LinearRegression()
model.fit(X_train, y_train)
print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_[0]}")
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

# Plot actual vs predicted future AAPL returns
plt.figure(figsize=(8, 6))
plt.scatter(X_test, y_test, color='blue', label='Actual Future AAPL Returns')
plt.plot(X_test, y_pred, color='red', label='Predicted Future AAPL Returns', linewidth=2)
plt.xlabel('Change in Volatility of the Market (VIX %)')  # VIX returns
plt.ylabel('Future AAPL Returns %')  # Future AAPL returns
plt.title(f'Linear Regression (Shifted by {shift_period} Period): VIX Returns vs Future AAPL Returns')
plt.legend()
plt.grid(True)
plt.show()

#We see that the relationship between volatility index and returns are not exactly causal, at least in the linear domain, which makes more since. The most likely reasin for the correlation between them is to be a third factor (Macro economic or geopolotical news) that causes both to correlate so much.

#Scenario 5: Bonds

In [ ]:
# Let's get US Government 10-Year Bond Yields
bond_rate = fred.get_series('IRLTLT01USM156N')

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(14, 7)
ax.plot(bond_rate.index, bond_rate.values)
ax.set_xlabel('Time')
ax.set_ylabel('Rate (%)')
ax.set_title('US Government 10-Year Bond Yields')



> Ignore low time frame movement, we can see bond yield rising from the 1950s and peaked in 1981, then gradually decrease until 2020, which then rise again until now. Let's look at US CPI and see if inflation has any impact on this change.



In [ ]:
# Get Sticky Price Consumer Price Index (CPI) less Food and Energy, which better incorporates future inflation
cpi = fred.get_series('CORESTICKM159SFRBATL')

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(14, 7)
ax.plot(cpi.index, cpi.values, label="CPI")
ax.plot(bond_rate.index, bond_rate.values, label="Bond Yield")
ax.set_xlabel('Time')
ax.set_ylabel('Rate (%)')
ax.set_title('Sticky Price Consumer Price Index (CPI) less Food and Energy')
ax.legend()


> As expected, the US government bond yields change with central bank interest rates, which is adjusted frequently as a primary tool to manage inflation

#Scenario 6: Options for AAPL far in/out of the money


> Let's see if Apple far ITM/OTM options are illiquid, and how VIX affect their liquidity

In [ ]:
vix = yf.download("^VIX", period="3mo", interval="1d")
aapl = yf.download('aapl', period="3mo", interval="1d")
# We only need quote from the last 1.5 month
vix = vix[len(vix) // 2:]
aapl = aapl[len(aapl) // 2:]

In [ ]:
aapl['ret'] = aapl['Adj Close'].pct_change() * 100
vix['ret'] = vix['Adj Close'].pct_change() * 100

In [ ]:
calls_aapl = options.get_calls("aapl")
calls_aapl.head()

In [ ]:
# AAPL's price as of Sep 26 is 227, let's consider only deep ITM/OTM options with strike difference from AAPL price of at least 10%
cur_price = 227
calls_aapl = calls_aapl[(calls_aapl["Strike"] > cur_price * 1.1) | (calls_aapl["Strike"] < cur_price * 0.9)]

In [ ]:
plt.hist(calls_aapl['Volume'].sort())
plt.xlabel('AAPL options volume')
plt.ylabel('Frequency')
plt.title('AAPL options volume distribution')
plt.show()

NameError: name 'plt' is not defined



> It can be seen that deep ITM/OTM options is very illiquid



In [ ]:
# We will sum up volume of all deep ITM/OTM options
calls_option = calls_aapl["Contract Name"]
aapl_combine_calls = pd.Series(0, index=aapl.index)
for option in calls_option:
  try:
    _call = yf.download(option, period="max", interval="1d")
    _call = pd.Series(0, index=aapl.index) + _call["Volume"]
    _call.fillna(0, inplace=True)
    aapl_combine_calls += _call
  except:
    pass

In [ ]:
# Let's plot options volume with VIX
fig, ax1 = plt.subplots()
fig.set_size_inches(14, 7)

ax1.plot(aapl_combine_calls.index, aapl_combine_calls.values, color='b', label='AAPL Option Volume')
ax1.set_xlabel('Time')
ax1.set_ylabel('Volume', color='b')
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(vix.index, vix["ret"], color='r', label='VIX')
ax2.set_ylabel('VIX', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax1.set_title('Illiquid options and VIX')


> It can be observerd that VIX is a good indicator of the illiquidity of Apple call options: When VIX increases, the asset becomes illiquid, and vice versa. The data is still limited with less than 1 month of data of only call options of one underlying asset. We need to thoroughly study other illiquid asset in a longer span of time to make a conclusion.

## References

Board of Governors of the Federal Reserve System (US), Commercial Bank Interest

1. Rate on Credit Card Plans, All Accounts [TERMCBCCALLNS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/TERMCBCCALLNS, September 26, 2024.
2. Board of Governors of the Federal Reserve System (US), Federal Funds Effective Rate [FEDFUNDS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/FEDFUNDS, September 26, 2024.
3. Freddie Mac, 15-Year Fixed Rate Mortgage Average in the United States [MORTGAGE15US], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/MORTGAGE15US, September 26, 2024.
4. Board of Governors of the Federal Reserve System (US), Delinquency Rate on Single-Family Residential Mortgages, Booked in Domestic Offices, All Commercial Banks [DRSFRMACBS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DRSFRMACBS, September 26, 2024.
5. Board of Governors of the Federal Reserve System (US), Delinquency Rate on Business Loans, All Commercial Banks [DRBLACBS], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DRBLACBS, September 26, 2024.
6. Organization for Economic Co-operation and Development, Interest Rates: Long-Term Government Bond Yields: 10-Year: Main (Including Benchmark) for United States [IRLTLT01USM156N], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/IRLTLT01USM156N, September 26, 2024.
7. Federal Reserve Bank of Atlanta, Sticky Price Consumer Price Index less Food and Energy [CORESTICKM159SFRBATL], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/CORESTICKM159SFRBATL, September 26, 2024.
8. Yahoo Finance, CBOE Volatility Index, https://finance.yahoo.com/quote/%5EVIX/, September 26, 2024.
9. Yahoo Finance, SPDR S&P 500 ETF Trust, https://finance.yahoo.com/quote/SPY/, September 26, 2024.
10. Yahoo Finance, Apple Inc., https://finance.yahoo.com/quote/AAPL/, September 26, 2024.
11. Yahoo Finance, Apple Options maturity Sep 27, https://finance.yahoo.com/quote/AAPL/options/?date=1727395200, September 26, 2024